In [3]:
# Importera biblioteket requests, som används för att hämta data från internet
import requests

# Adressen till Open-Meteos API för historiskt väder
url = "https://archive-api.open-meteo.com/v1/archive"

# Ange vilken information vi vill hämta
params = {
    "latitude": 37.5665,          # Seoul
    "longitude": 126.9780,        # Seoul
    "start_date": "2018-06-01",   # Startdatum
    "end_date": "2018-06-03",     # Slutdatum
    "hourly": "cloud_cover",      # Molnighet per timme
    "timezone": "Asia/Seoul"      # Lokal tid i Seoul
}

# Skicka en förfrågan till API:et
response = requests.get(url, params=params, timeout=30)

# Visa statuskoden för att kontrollera om anropet lyckades
print("HTTP-status:", response.status_code)

HTTP-status: 503


In [4]:
# Importera pandas för att arbeta med data i tabellform
import pandas as pd

# Omvandla API-svaret från JSON till ett Python-dictionary
data = response.json()

# Skapa en DataFrame med väderinformationen
weather_df = pd.DataFrame(data["hourly"])

# Visa de fem första raderna
weather_df.head()

KeyError: 'hourly'

In [ ]:
# Kontrollera hur många rader och kolumner vi fått
print("Antal rader och kolumner:", weather_df.shape)

# Kontrollera om någon information saknas
print("\nSaknade värden:")
print(weather_df.isnull().sum())

# Visa de fem sista observationerna
display(weather_df.tail())

Antal rader och kolumner: (72, 2)

Saknade värden:
time           0
cloud_cover    0
dtype: int64


,time,cloud_cover
67,2018-06-03T19:00,0
68,2018-06-03T20:00,0
69,2018-06-03T21:00,57
70,2018-06-03T22:00,11
71,2018-06-03T23:00,76


In [ ]:
import pandas as pd

# Läs in originaldatasetet med cykeluthyrningar
# Filen använder en annan teckenkodning än UTF-8
# Läs originaldata från undermappen data
bike_df = pd.read_csv("data/SeoulBikeData.csv", encoding="latin1")

# Visa hur många rader och kolumner datasetet innehåller
print("Antal rader och kolumner:", bike_df.shape)

# Visa de fem första raderna
display(bike_df.head())

Antal rader och kolumner: (8760, 14)


,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day
0,01/12/2017,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


In [ ]:
# Omvandla BikeFlows datum från text till ett riktigt datum
# dayfirst=True betyder att dagen kommer före månaden
bike_df["Date"] = pd.to_datetime(
    bike_df["Date"],
    format="%d/%m/%Y"
)

# Lägg till timmen för att skapa en fullständig tidsstämpel
bike_df["datetime"] = (
    bike_df["Date"] + pd.to_timedelta(bike_df["Hour"], unit="h")
)

# Omvandla Open-Meteos tidskolumn till samma datatyp
weather_df["datetime"] = pd.to_datetime(weather_df["time"])

# Visa de första raderna från båda tabellerna
print("BikeFlow:")
display(bike_df[["Date", "Hour", "datetime"]].head())

print("Open-Meteo:")
display(weather_df[["time", "datetime", "cloud_cover"]].head())

BikeFlow:


,Date,Hour,datetime
0,2017-12-01,0,2017-12-01 00:00:00
1,2017-12-01,1,2017-12-01 01:00:00
2,2017-12-01,2,2017-12-01 02:00:00
3,2017-12-01,3,2017-12-01 03:00:00
4,2017-12-01,4,2017-12-01 04:00:00


Open-Meteo:


,time,datetime,cloud_cover
0,2018-06-01T00:00,2018-06-01 00:00:00,0
1,2018-06-01T01:00,2018-06-01 01:00:00,0
2,2018-06-01T02:00,2018-06-01 02:00:00,0
3,2018-06-01T03:00,2018-06-01 03:00:00,0
4,2018-06-01T04:00,2018-06-01 04:00:00,0


In [ ]:
# Slå ihop BikeFlow med molnigheten från Open-Meteo
bike_merged = pd.merge(
    bike_df,                          # Vårt huvuddataset med cykeluthyrningar
    weather_df[["datetime", "cloud_cover"]],  # Extern väderdata
    on="datetime",                    # Matcha på datum och klockslag
    how="left"                        # Behåll alla rader från BikeFlow
)

# Kontrollera hur många rader som fick molnighetsdata
print("Antal matchade rader:", bike_merged["cloud_cover"].notna().sum())

# Visa hur många rader vi har totalt
print("Totalt antal rader:", len(bike_merged))

# Visa de första matchade raderna
display(
    bike_merged[bike_merged["cloud_cover"].notna()][
        ["datetime", "Rented Bike Count", "cloud_cover"]
    ].head()
)

Antal matchade rader: 72
Totalt antal rader: 8760


,datetime,Rented Bike Count,cloud_cover
4368,2018-06-01 00:00:00,963,0.0
4369,2018-06-01 01:00:00,776,0.0
4370,2018-06-01 02:00:00,554,0.0
4371,2018-06-01 03:00:00,366,0.0
4372,2018-06-01 04:00:00,265,0.0


In [ ]:
# Hämta första och sista datumet från BikeFlow
start_date = bike_df["Date"].min().strftime("%Y-%m-%d")
end_date = bike_df["Date"].max().strftime("%Y-%m-%d")

# Uppdatera datumen i våra API-parametrar
params["start_date"] = start_date
params["end_date"] = end_date

# Hämta molnighet för hela perioden
response_full = requests.get(url, params=params, timeout=60)

# Kontrollera att API-anropet lyckades
response_full.raise_for_status()

# Omvandla API-svaret till en DataFrame
weather_full = pd.DataFrame(response_full.json()["hourly"])

# Kontrollera resultatet
print("Antal väderobservationer:", len(weather_full))
print("Saknade värden:")
print(weather_full.isnull().sum())

# Visa de första raderna
display(weather_full.head())

Antal väderobservationer: 8760
Saknade värden:
time           0
cloud_cover    0
dtype: int64


,time,cloud_cover
0,2017-12-01T00:00,0
1,2017-12-01T01:00,0
2,2017-12-01T02:00,0
3,2017-12-01T03:00,0
4,2017-12-01T04:00,0


In [ ]:
# Omvandla Open-Meteos tidskolumn till datetime
weather_full["datetime"] = pd.to_datetime(weather_full["time"])

# Slå ihop cykeldata med hela årets molnighetsdata
bike_final = pd.merge(
    bike_df,
    weather_full[["datetime", "cloud_cover"]],
    on="datetime",
    how="left",
    validate="one_to_one"  # Kontrollera att varje tidpunkt är unik i båda tabellerna
)

# Kontrollera resultatet
print("Antal rader:", len(bike_final))
print("Matchade rader:", bike_final["cloud_cover"].notna().sum())
print("Saknade värden i molnighet:", bike_final["cloud_cover"].isna().sum())

# Visa några kolumner från den sammanslagna tabellen
display(
    bike_final[["datetime", "Rented Bike Count", "cloud_cover"]].head()
)

Antal rader: 8760
Matchade rader: 8760
Saknade värden i molnighet: 0


,datetime,Rented Bike Count,cloud_cover
0,2017-12-01 00:00:00,254,0
1,2017-12-01 01:00:00,204,0
2,2017-12-01 02:00:00,173,0
3,2017-12-01 03:00:00,107,0
4,2017-12-01 04:00:00,78,0


In [ ]:
# Spara väderdatan som hämtats från Open-Meteo
weather_full.to_csv("data/open_meteo_seoul.csv", index=False)

# Spara BikeFlow med den nya kolumnen cloud_cover
bike_final.to_csv("data/bikeflow_enriched.csv", index=False)

# Bekräfta att filerna har skapats
print("Filerna har sparats!")

Filerna har sparats!


In [ ]:
from pathlib import Path

# Visa vilken mapp Python arbetar i
print("Arbetsmapp:", Path.cwd())

# Kontrollera om Python hittar våra tre CSV-filer
print("Originaldata:", Path("data/SeoulBikeData.csv").exists())
print("Väderdata:", Path("data/open_meteo_seoul.csv").exists())
print("Berikad data:", Path("data/bikeflow_enriched.csv").exists())

Arbetsmapp: /Users/annadontchev/BikeFlow
Originaldata: True
Väderdata: True
Berikad data: True


In [5]:
# Läs in den färdigberikade datan
df = pd.read_csv("data/bikeflow_enriched.csv")

# Kontrollera storleken
print("Antal rader och kolumner:", df.shape)

# Visa de första raderna
display(df.head())

Antal rader och kolumner: (8760, 16)


,Date,Rented Bike Count,Hour,Temperature(°C),Humidity(%),Wind speed (m/s),Visibility (10m),Dew point temperature(°C),Solar Radiation (MJ/m2),Rainfall(mm),Snowfall (cm),Seasons,Holiday,Functioning Day,datetime,cloud_cover
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 00:00:00,0
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 01:00:00,0
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 02:00:00,0
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 03:00:00,0
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 04:00:00,0
